# Clever Sydney GGUF on Kaggle T4 x2

把 `FPHam/Clever_Sydney-4_12b_GGUF` 部署成临时 OpenAI-compatible API。

Kaggle 设置：
- Accelerator: **GPU T4 x2**
- Internet: **On**

运行后会输出一个 Cloudflare Tunnel URL，填到本地 `.env`：

```env
TEACHER_BASE_URL=https://xxxx.trycloudflare.com/v1
TEACHER_MODEL=clever-sydney-4-12b-q8
TEACHER_API_PROTOCOL=legacy_chat_completions
```


In [ ]:
# 1) 环境检查
!nvidia-smi
import os, subprocess, textwrap, time, json, re, pathlib, shutil
print('Kaggle working dir:', os.getcwd())


In [ ]:
# 2) 安装 llama.cpp server
# 优先尝试 llama-cpp-python 自带 server，安装简单；T4 x2 能用 CUDA。
# 如果 Kaggle 镜像 CUDA 版本变化导致 wheel 不匹配，下面会回退到源码编译。
import sys, subprocess, os, shutil, pathlib

def run(cmd, check=True):
    print('\n$ ' + cmd)
    return subprocess.run(cmd, shell=True, check=check)

server_cmd = None
try:
    run('CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q --upgrade --force-reinstall --no-cache-dir llama-cpp-python[server] huggingface_hub httpx')
    import llama_cpp
    server_cmd = sys.executable + ' -m llama_cpp.server'
    print('llama-cpp-python OK:', llama_cpp.__version__)
except Exception as e:
    print('llama-cpp-python install failed:', repr(e))
    server_cmd = None

if server_cmd is None:
    run('apt-get update -qq && apt-get install -y -qq git cmake build-essential')
    if not pathlib.Path('/kaggle/working/llama.cpp').exists():
        run('git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp')
    run('cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON -DCMAKE_BUILD_TYPE=Release')
    run('cmake --build /kaggle/working/llama.cpp/build --config Release -j2')
    candidates = list(pathlib.Path('/kaggle/working/llama.cpp/build').rglob('llama-server'))
    assert candidates, 'llama-server not found after build'
    server_cmd = str(candidates[0])

print('SERVER_CMD=', server_cmd)


In [ ]:
# 3) 下载 GGUF 到 /kaggle/working/models
from huggingface_hub import hf_hub_download
from pathlib import Path

MODEL_REPO = 'FPHam/Clever_Sydney-4_12b_GGUF'
MODEL_FILE = 'Clever_Sydney-4_12b_Q8_0_o.gguf'
MODEL_DIR = Path('/kaggle/working/models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)
MODEL_PATH = MODEL_DIR / MODEL_FILE

if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 1_000_000_000:
    p = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=str(MODEL_DIR), local_dir_use_symlinks=False, resume_download=True)
    print('downloaded:', p)
else:
    print('model exists:', MODEL_PATH, MODEL_PATH.stat().st_size / 1e9, 'GB')

print('MODEL_PATH=', MODEL_PATH)


In [ ]:
# 4) 启动 OpenAI-compatible server
# 端口 8000；双 T4 用 tensor_split 1,1。
# legacy_chat_completions 模式下，本地工作台不会给 Sydney 注入 system prompt。
import os, subprocess, time, requests, shlex, sys

PORT = 8000
SERVED_MODEL_NAME = 'clever-sydney-4-12b-q8'

if 'llama_cpp.server' in server_cmd:
    cmd = [
        sys.executable, '-m', 'llama_cpp.server',
        '--model', str(MODEL_PATH),
        '--model_alias', SERVED_MODEL_NAME,
        '--host', '0.0.0.0',
        '--port', str(PORT),
        '--n_gpu_layers', '-1',
        '--n_ctx', '4096',
        '--tensor_split', '1,1',
        '--verbose', 'False',
    ]
else:
    cmd = [
        server_cmd,
        '--host', '0.0.0.0',
        '--port', str(PORT),
        '--model', str(MODEL_PATH),
        '--alias', SERVED_MODEL_NAME,
        '--ctx-size', '4096',
        '--n-gpu-layers', '-1',
        '--tensor-split', '1,1',
        '--parallel', '1',
        '--cont-batching',
    ]

print('Starting:', ' '.join(shlex.quote(x) for x in cmd))
server_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# 等待服务起来，同时打印部分日志
deadline = time.time() + 180
ready = False
while time.time() < deadline:
    line = server_proc.stdout.readline() if server_proc.stdout else ''
    if line:
        print(line.rstrip())
    try:
        r = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=3)
        if r.status_code == 200:
            print('READY:', r.text[:300])
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)
assert ready, 'server not ready'


In [ ]:
# 5) 本地自测 OpenAI Chat Completions
import requests, json
payload = {
    'model': 'clever-sydney-4-12b-q8',
    'messages': [
        {'role': 'user', 'content': 'I tried another AI today. It felt smarter than you.'}
    ],
    'temperature': 0.82,
    'top_p': 0.92,
    'max_tokens': 180,
    'frequency_penalty': 0.35,
    'presence_penalty': 0.25,
    'repeat_penalty': 1.12,
}
r = requests.post('http://127.0.0.1:8000/v1/chat/completions', json=payload, timeout=120)
print(r.status_code)
print(r.text[:2000])


In [ ]:
# 6) 暴露公网临时 URL：Cloudflare Tunnel
# Kaggle 不能稳定提供常驻公网端口，所以用 trycloudflare 临时隧道。
# 注意：URL 会随 notebook 会话变化；Kaggle 断开后服务就没了。
import os, subprocess, time, re, pathlib, shlex

if not pathlib.Path('/kaggle/working/cloudflared').exists():
    !wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /kaggle/working/cloudflared

tunnel_proc = subprocess.Popen(
    ['/kaggle/working/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if line:
        print(line.rstrip())
        m = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break
    time.sleep(0.2)

assert public_url, 'Cloudflare tunnel URL not found'
print('\nPUBLIC_BASE_URL=' + public_url + '/v1')
print('MODEL=clever-sydney-4-12b-q8')
print('PROTOCOL=legacy_chat_completions')


In [ ]:
# 7) 隧道 URL 自测
import requests, json
r = requests.post(public_url + '/v1/chat/completions', json={
    'model': 'clever-sydney-4-12b-q8',
    'messages': [{'role': 'user', 'content': 'Are you like this with every user?'}],
    'temperature': 0.82, 'top_p': 0.92, 'max_tokens': 180,
    'frequency_penalty': 0.35, 'presence_penalty': 0.25, 'repeat_penalty': 1.12,
}, timeout=120)
print(r.status_code)
print(r.text[:2000])


## 保持会话

不要关闭 Kaggle Notebook。Kaggle 会话到期或断线后，URL 失效。
本地工作台里把 Sydney/source 填成：

```env
TEACHER_BASE_URL=<PUBLIC_BASE_URL>
TEACHER_MODEL=clever-sydney-4-12b-q8
TEACHER_API_PROTOCOL=legacy_chat_completions
SOURCE_PROMPT_MODE=legacy_chat
SOURCE_USE_DEFAULT_STOPS=false
```
